# Evaluation

Measuring the pipeline defined in `01_pipeline_and_demo.ipynb` against a stored ground
truth, following a five-step process:

| Step | What it means |
|---|---|
| **1. Build a test set** | Real documents, digital *and* scanned |
| **2. Define ground truth** | A stored answer key in a file, not buried in code |
| **3. Pick metrics** | A deliberate choice of what matters |
| **4. Run the evaluation** | Feed the test set through, compare against the key |
| **5. Show results clearly** | One table readable at a glance |

Retrieval and answer generation are measured separately throughout. If an answer is wrong,
we need to know whether the passage was never found, or was found and then misread.

**Requires:** `pharma-blob-sample.pdf`, `sample-sdf-document.pdf` and `ground_truth.json`
in `/content`. The scanned copies are generated automatically.

In [ ]:
# ============================================
# Load the pipeline
# ============================================
# This notebook evaluates the pipeline defined in 01_pipeline_and_demo.ipynb.
# Rather than duplicating that code, run the pipeline notebook's definitions
# here. In Colab, put both notebooks in the same folder and run:
#
#   %run 01_pipeline_and_demo.ipynb
#
# ...or simply copy Steps 1-9 from that notebook into the cells below.
# The final Gradio launch cell is not needed for evaluation.
# ============================================

# %run 01_pipeline_and_demo.ipynb
print("Run the pipeline notebook's Steps 1-9 before continuing.")

## Steps 1 & 2 — Test set and ground truth

One file does both jobs: `ground_truth.json` declares which documents make up the test set
*and* holds the answer key for them.

The test set includes **scanned copies** of both source PDFs. Both originals carry a full
text layer, so without them the OCR half of the pipeline would never run once and its
measured cost would be zero. The scanned copies are rasterised from the originals, giving
matched pairs — same content, different input format — so any accuracy gap between them is
caused by OCR rather than by different documents.

**The answer key holds** 13 answerable questions with keywords taken from the actual
document text, plus **4 unanswerable control questions** whose answers are genuinely absent
from the corpus. The strongest is Q11, which asks for the lot number of part 29477427: that
part appears in five of the seven documents, but its Certificate of Quality is missing from
the bundle, so a model that pattern-matches will borrow a lot number from a neighbouring
certificate. Without controls you cannot tell a system that knows things from one that
makes things up.

Every keyword was verified against the extracted text of the page it is attributed to, in
both the digital PDFs and the OCR output of the scanned copies.

In [ ]:
# ============================================
# EVAL STEP 2: Load the ground truth answer key
# ============================================
#
# WHAT WE'RE DOING:
# Loading the test set definition from ground_truth.json -- a file that
# lives outside this notebook. It defines the corpus documents, the test
# questions, which document type each answer lives in, which questions are
# deliberately unanswerable, and (optionally) the keywords a correct
# answer must contain.
#
# WHY THIS MATTERS:
# The answer key used to be a Python list inside the notebook. Keeping it
# in a separate JSON file means it can be edited, reviewed and version
# controlled without touching pipeline code, extended by a domain expert
# who does not read Python, and reused by any other evaluation script.
# It also makes the separation explicit: the notebook is the system under
# test, the JSON is the yardstick.
#
# If the file is missing, the cell writes a starter copy so the notebook
# is runnable from a clean checkout.
#
# WHAT YOU'LL SEE:
# A summary of the loaded test set, a breakdown by question category, and
# a warning telling you how many questions still need answer keywords
# filled in.
# ============================================

GROUND_TRUTH_PATH = "/content/ground_truth.json"

STARTER_GROUND_TRUTH = {
    "dataset_name": "Pharma Document Q&A - Ground Truth Test Set",
    "version": "1.0",
    "documents": [],
    "questions": [],
}


def load_ground_truth(path: str = GROUND_TRUTH_PATH) -> Dict:
    """Read the answer key, writing a starter file if none exists."""
    if not os.path.exists(path):
        print(f"No ground truth found at {path} -- writing a starter file.")
        print("Upload the provided ground_truth.json to /content and re-run this cell.")
        with open(path, "w") as f:
            json.dump(STARTER_GROUND_TRUTH, f, indent=2)

    with open(path) as f:
        gt = json.load(f)

    # Validate rather than trust: a malformed key produces silently wrong
    # metrics, which is worse than an error.
    required = {"id", "question", "source_file", "relevant_doc_types",
                "retrieval_graded", "answerable"}
    for q in gt.get("questions", []):
        missing = required - set(q)
        if missing:
            raise ValueError(f"Question {q.get('id', '?')} is missing fields: {sorted(missing)}")
        if q["retrieval_graded"] and not q["relevant_doc_types"]:
            raise ValueError(f"Question {q['id']} is marked retrieval_graded but lists no relevant_doc_types")

    return gt


GROUND_TRUTH = load_ground_truth()
QUESTIONS = GROUND_TRUTH["questions"]

if not QUESTIONS:
    raise RuntimeError("Ground truth contains no questions. Upload the full ground_truth.json to /content.")

gt_summary = pd.DataFrame([{
    "ID": q["id"],
    "Category": q.get("category", "-"),
    "Source file": q["source_file"],
    "Relevant doc types": ", ".join(q["relevant_doc_types"]) or "-",
    "Answerable": q["answerable"],
    "Keywords set": len(q.get("expected_answer_keywords", [])),
} for q in QUESTIONS])

print(f"Loaded '{GROUND_TRUTH['dataset_name']}' v{GROUND_TRUTH.get('version', '?')}")
print(f"  {len(QUESTIONS)} questions across {len(GROUND_TRUTH['documents'])} documents")
print(f"  {sum(q['answerable'] for q in QUESTIONS)} answerable, "
      f"{sum(not q['answerable'] for q in QUESTIONS)} unanswerable controls")
print(f"  {sum(q['retrieval_graded'] for q in QUESTIONS)} graded for retrieval\n")
display(gt_summary)

n_unkeyed = sum(1 for q in QUESTIONS
                if q["answerable"] and not q.get("expected_answer_keywords"))
if n_unkeyed:
    print(f"\nNOTE: {n_unkeyed} answerable question(s) have no expected_answer_keywords yet.")
    print("Retrieval, citation, abstention and faithfulness metrics do not need them and will")
    print("run regardless. Keyword-based answer accuracy will report those questions as")
    print("UNGRADED rather than scoring them zero. Run bootstrap_answer_key() after the")
    print("pipeline has processed the corpus to draft them from the real document text.")

In [ ]:
# ============================================
# EVAL STEP 1: Build the test set (digital + scanned)
# ============================================
#
# WHAT WE'RE DOING:
# Taking the two source PDFs and generating an image-only ("scanned")
# copy of each by rasterising every page and re-embedding it as a picture
# with no text layer. Optional light degradation -- a small rotation and
# a touch of noise -- imitates a real scan rather than a clean render.
#
# WHY THIS MATTERS:
# The brief asks for a test set mixing digital and scanned documents. Both
# source PDFs carry a full text layer, so before this cell the OCR branch
# of the pipeline never executed even once: measured OCR time was 0.0s and
# OCR was effectively untested. Rasterising the same documents gives us
# matched pairs -- identical content, different input modality -- which is
# a stronger test than two unrelated files, because any drop in accuracy
# between a pair is attributable to OCR rather than to the content.
#
# HONEST LIMITATION: these are synthetic scans. They exercise the OCR code
# path and expose OCR's cost and error rate, but a genuine scan (skew,
# shadows, coffee stains, stamps, handwriting) would be harsher. This is
# noted in the results rather than glossed over.
#
# WHAT YOU'LL SEE:
# A confirmation line per generated file, then a table of the four-file
# corpus with page counts and file sizes.
# ============================================

# Scan quality presets. "moderate" is the default: clearly a scan (low
# resolution, visible skew, sensor noise, JPEG artefacts) while still
# preserving the text well enough that a difference in results points at the
# pipeline rather than at text that OCR simply lost.
#
# Measured on these documents: at "moderate", Tesseract recovers ~100% of the
# words and every answer-key fact survives. At "heavy" it drops ~10% of words
# and loses about a third of the answer-key facts outright -- at that point the
# evaluation is measuring OCR failure rather than the pipeline, which is why it
# is not the default. Switch to "heavy" if you want a deliberate stress test.
SCAN_PRESETS = {
    "clean":    dict(dpi=150, rotation=0.4, noise=6,  blur=0.0, quality=80),
    "light":    dict(dpi=120, rotation=0.8, noise=10, blur=0.3, quality=65),
    "moderate": dict(dpi=100, rotation=1.2, noise=16, blur=0.5, quality=55),
    "heavy":    dict(dpi=85,  rotation=1.8, noise=22, blur=0.8, quality=45),
}
SCAN_QUALITY = "moderate"


def make_scanned_copy(src_path: str, dst_path: str, preset: str = None) -> str:
    """
    Render every page of src_path to an image and write an image-only PDF to
    dst_path. The result has no text layer, so the pipeline must OCR it.

    Pages are saved as JPEG rather than PNG: it is what a real scanner
    produces, the compression artefacts are part of what makes this a
    realistic test, and it keeps the file ~10x smaller.
    """
    cfg = SCAN_PRESETS[preset or SCAN_QUALITY]
    src = fitz.open(src_path)
    out = fitz.open()
    rng = np.random.default_rng(0)

    for page in src:
        pix = page.get_pixmap(dpi=cfg["dpi"])
        img = Image.open(io.BytesIO(pix.tobytes("png"))).convert("L")

        if cfg["rotation"]:
            img = img.rotate(cfg["rotation"], resample=Image.BICUBIC, fillcolor=255)
        if cfg["blur"]:
            img = img.filter(ImageFilter.GaussianBlur(cfg["blur"]))
        if cfg["noise"]:
            arr = np.asarray(img).astype(np.int16)
            arr += rng.integers(-cfg["noise"], cfg["noise"] + 1,
                                (img.height, img.width), dtype=np.int16)
            img = Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))

        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=cfg["quality"])

        new_page = out.new_page(width=page.rect.width, height=page.rect.height)
        new_page.insert_image(new_page.rect, stream=buf.getvalue())

    out.save(dst_path, deflate=True)
    out.close()
    src.close()
    return dst_path


def build_test_corpus(gt_documents: List[Dict], base_dir: str = "/content") -> pd.DataFrame:
    """
    Ensure every file listed in the ground truth's `documents` section
    exists on disk, generating the scanned variants from their sources.
    Returns a summary table of the corpus.
    """
    rows = []
    for entry in gt_documents:
        path = os.path.join(base_dir, entry["file"])

        if entry["variant"] == "scanned" and not os.path.exists(path):
            src = os.path.join(base_dir, entry["derived_from"])
            if not os.path.exists(src):
                print(f"  SKIP {entry['file']}: source {entry['derived_from']} not found in {base_dir}")
                continue
            print(f"  Generating {entry['file']} from {entry['derived_from']} (scan quality: {SCAN_QUALITY})...")
            make_scanned_copy(src, path)

        if not os.path.exists(path):
            print(f"  MISSING {entry['file']} - upload it to {base_dir}")
            continue

        with fitz.open(path) as d:
            n_pages = d.page_count
            text_chars = sum(len(p.get_text().strip()) for p in d)

        rows.append({
            "File": entry["file"],
            "Variant": entry["variant"].title(),
            "Pages": n_pages,
            "Size (KB)": round(os.path.getsize(path) / 1024, 1),
            "Text-layer chars": text_chars,
            "Has text layer": "yes" if text_chars > 100 else "no (OCR required)",
            "Expected doc types": len(entry["expected_doc_types"]),
            "path": path,
        })

    return pd.DataFrame(rows)


BASE_DIR = "/content"  # change to "." if running outside Colab

corpus_df = build_test_corpus(GROUND_TRUTH["documents"], base_dir=BASE_DIR)
print("\nTest corpus:")
display(corpus_df.drop(columns=["path"]))

if corpus_df.empty:
    raise RuntimeError(
        f"No test documents found in {BASE_DIR}. Upload pharma-blob-sample.pdf and "
        f"sample-sdf-document.pdf there, then re-run this cell.")

## Step 3 — Choosing the metrics

**Retrieval** — did we find the right passage? Hit Rate, Recall@K, Precision@K, MRR.

**Answer accuracy** — was the answer right? Keyword coverage against the answer key, plus
correct refusal on the control questions.

**Citations** — can a reader verify it? Split into *validity* (was the cited document
actually retrieved) and *correctness* (was it the right one). Separate metrics, because an
answer can cite a real-but-wrong source, which looks entirely trustworthy on the page.

**Speed** — response time broken into OCR, retrieval and generation, so a slow answer can
be traced to the stage responsible.

In [ ]:
# ============================================
# EVAL STEP 3: Metric definitions
# ============================================
#
# WHAT WE'RE DOING:
# Defining the scoring functions used to grade the pipeline. Retrieval is
# graded with four standard information-retrieval metrics (Hit Rate,
# Recall@K, Precision@K, MRR). Answers are graded on keyword coverage
# against the stored answer key, on correct abstention for the
# deliberately unanswerable control questions, and on whether the
# citations the model emitted are both valid and correct.
#
# WHY THIS MATTERS:
# These are deliberate choices, not just whatever was easiest to compute.
# See the markdown cell above for the rationale behind each one. Defining
# them as pure functions (no pipeline state, no model calls) means they
# can be reasoned about and unit-tested independently of the RAG system.
#
# WHAT YOU'LL SEE:
# A short self-test confirming each metric returns the expected value on
# a hand-worked example.
# ============================================

import re
from typing import List, Dict, Optional, Sequence


# ------------------------------------------------------------------
# Relevance judgement
# ------------------------------------------------------------------
# A retrieved chunk counts as relevant if it came from a document type
# the answer key lists as containing the answer. Document-type-level
# relevance is used (rather than chunk-level) because the answer key is
# annotated at the document level -- this is stated in the results so the
# metric is not over-claimed.

def is_relevant(chunk_doc_type: str, relevant_doc_types: Sequence[str]) -> bool:
    """True if a retrieved chunk's document type is one the answer key marks relevant."""
    return chunk_doc_type in set(relevant_doc_types)


def retrieval_metrics(retrieved_doc_types: List[str],
                      relevant_doc_types: Sequence[str],
                      total_relevant_chunks: int,
                      k: int) -> Dict:
    """
    Compute Hit Rate, Precision@K, Recall@K and Reciprocal Rank for a
    single query.

    retrieved_doc_types  -- doc_type of each retrieved chunk, in rank order
    relevant_doc_types   -- doc types the answer key marks as containing the answer
    total_relevant_chunks-- how many chunks in the whole corpus are relevant
                            (denominator for Recall@K)
    k                    -- cutoff; only the first k results are scored
    """
    top_k = retrieved_doc_types[:k]
    flags = [is_relevant(dt, relevant_doc_types) for dt in top_k]
    n_hits = sum(flags)

    # Reciprocal rank: 1/(rank of first relevant result), 0 if none found.
    rr = 0.0
    for rank, flag in enumerate(flags, start=1):
        if flag:
            rr = 1.0 / rank
            break

    # Recall@K is capped at 1.0: retrieving k chunks can never surface more
    # than k of the relevant ones, so we divide by the achievable maximum.
    achievable = min(total_relevant_chunks, k) if total_relevant_chunks else 0
    recall = (n_hits / achievable) if achievable else float("nan")

    return {
        "hit": bool(n_hits > 0),
        "precision_at_k": n_hits / k if k else 0.0,
        "recall_at_k": recall,
        "reciprocal_rank": rr,
        "n_relevant_retrieved": n_hits,
        "total_relevant_in_corpus": total_relevant_chunks,
    }


# ------------------------------------------------------------------
# Answer grading
# ------------------------------------------------------------------

def _normalise(text: str) -> str:
    """Lowercase and collapse punctuation/whitespace so keyword matching is robust."""
    return re.sub(r"[^a-z0-9]+", " ", (text or "").lower()).strip()


# NOTE: these patterns are matched against text that has already been put
# through _normalise(), i.e. lowercase with all punctuation collapsed to
# single spaces. That is why "doesn't" is written here as "doesn t" and why
# the patterns contain no punctuation of their own.
ABSTENTION_PATTERNS = [
    r"\b(?:does|do|did) not (?:contain|provide|specify|include|mention|state)\b",
    r"\b(?:doesn|don) t (?:contain|provide|specify|include|mention|state)\b",
    r"\bnot (?:contained|provided|specified|mentioned|found|available|included|stated)\b",
    r"\bno (?:information|mention|reference|details)\b",
    r"\bcannot (?:be )?(?:determine|determined|answer|answered|find|found)\b",
    r"\bcan t (?:determine|answer|find)\b",
    r"\bunable to (?:determine|answer|find|locate)\b",
    r"\b(?:could|can) not find\b",
    r"\b(?:couldn|can) t find\b",
    r"\bnot able to (?:determine|answer|find)\b",
    r"\binsufficient (?:information|context|detail)\b",
    r"\bis not (?:in|present|available)\b",
    r"\bisn t (?:in|present|available)\b",
]


def is_abstention(answer: str) -> bool:
    """True if the answer declines to answer rather than inventing content."""
    norm = _normalise(answer)
    return any(re.search(p, norm) for p in ABSTENTION_PATTERNS)


def keyword_coverage(answer: str, keywords: Sequence[str]) -> float:
    """
    Fraction of answer-key keywords present in the answer.
    Returns nan when no keywords are annotated, so unfilled rows are
    excluded from the accuracy average rather than silently scored zero.
    """
    if not keywords:
        return float("nan")
    norm = _normalise(answer)
    found = sum(1 for kw in keywords if _normalise(kw) and _normalise(kw) in norm)
    return found / len(keywords)


def grade_answer(answer: str,
                 keywords: Sequence[str],
                 answerable: bool = True,
                 threshold: float = 0.6) -> Dict:
    """
    Grade one answer.

    Answerable questions are scored on keyword coverage against the answer
    key. Unanswerable control questions are scored on whether the system
    correctly abstained -- an invented answer there is a hallucination and
    is marked incorrect.
    """
    if not answerable:
        correct = is_abstention(answer)
        return {
            "coverage": float("nan"),
            "correct": correct,
            "graded": True,
            "grading_mode": "abstention",
        }

    cov = keyword_coverage(answer, keywords)
    if cov != cov:  # nan -- answer key not filled in for this question
        return {
            "coverage": float("nan"),
            "correct": None,
            "graded": False,
            "grading_mode": "ungraded (no answer key)",
        }
    return {
        "coverage": cov,
        "correct": bool(cov >= threshold),
        "graded": True,
        "grading_mode": "keyword",
    }


# ------------------------------------------------------------------
# Citation grading
# ------------------------------------------------------------------

CITATION_RE = re.compile(r"\[From\s+([^,\]]+?)\s*(?:,\s*Pages?\s*([0-9]+)\s*-\s*([0-9]+))?\s*(?:,[^\]]*)?\]",
                         re.IGNORECASE)


def parse_citations(answer: str) -> List[Dict]:
    """Pull [From <DocType>, Pages A-B] markers out of a generated answer."""
    out = []
    for m in CITATION_RE.finditer(answer or ""):
        out.append({
            "doc_type": m.group(1).strip(),
            "page_start": int(m.group(2)) if m.group(2) else None,
            "page_end": int(m.group(3)) if m.group(3) else None,
        })
    return out


def grade_citations(answer: str,
                    retrieved_doc_types: Sequence[str],
                    relevant_doc_types: Sequence[str]) -> Dict:
    """
    Two separate citation checks:

    valid   -- every document type the answer cites was actually in the
               retrieved context. A citation to something that was never
               retrieved is a fabricated source.
    correct -- at least one cited document type matches the answer key.
               An answer can cite a real-but-wrong source; this catches it.
    """
    cites = parse_citations(answer)
    cited_types = {c["doc_type"].lower() for c in cites}
    retrieved_set = {d.lower() for d in retrieved_doc_types}
    relevant_set = {d.lower() for d in relevant_doc_types}

    if not cited_types:
        return {
            "has_citation": False,
            "citations_valid": None,
            "citations_correct": None,
            "n_citations": 0,
            "cited_types": [],
        }

    return {
        "has_citation": True,
        "citations_valid": cited_types.issubset(retrieved_set),
        "citations_correct": bool(cited_types & relevant_set) if relevant_set else None,
        "n_citations": len(cites),
        "cited_types": sorted(cited_types),
    }


# ------------------------------------------------------------------
# Self-test
# ------------------------------------------------------------------
def _selftest():
    # Retrieval: relevant chunk at rank 2 of 4, 3 relevant chunks in corpus.
    m = retrieval_metrics(["Cover Letter", "Certificate Of Quality", "Other", "Other"],
                          ["Certificate Of Quality"], total_relevant_chunks=3, k=4)
    assert m["hit"] is True
    assert abs(m["reciprocal_rank"] - 0.5) < 1e-9
    assert abs(m["precision_at_k"] - 0.25) < 1e-9
    assert abs(m["recall_at_k"] - (1 / 3)) < 1e-9

    # No relevant result retrieved.
    m2 = retrieval_metrics(["Other", "Other"], ["Cover Letter"], total_relevant_chunks=2, k=2)
    assert m2["hit"] is False and m2["reciprocal_rank"] == 0.0

    # Recall denominator capped by k.
    m3 = retrieval_metrics(["A", "A"], ["A"], total_relevant_chunks=10, k=2)
    assert abs(m3["recall_at_k"] - 1.0) < 1e-9

    # Abstention detection.
    assert is_abstention("The provided context does not contain that information.")
    assert is_abstention("I am unable to determine this from the documents.")
    assert not is_abstention("The lot number is 12345678 and it expires in 2027.")

    # Keyword grading.
    g = grade_answer("The lot number is 12345678, expiring 2027-05-01.",
                     ["12345678", "2027"], answerable=True)
    assert g["correct"] is True and abs(g["coverage"] - 1.0) < 1e-9
    g2 = grade_answer("It is a flow kit.", ["12345678", "2027"], answerable=True)
    assert g2["correct"] is False
    g3 = grade_answer("anything", [], answerable=True)
    assert g3["graded"] is False and g3["correct"] is None
    g4 = grade_answer("The context does not provide this.", [], answerable=False)
    assert g4["correct"] is True

    # Citations.
    ans = "Storage is 2-8C [From Cover Letter, Pages 1-1] per the letter."
    c = grade_citations(ans, ["Cover Letter", "Certificate Of Quality"], ["Cover Letter"])
    assert c["has_citation"] and c["citations_valid"] and c["citations_correct"]

    bad = "See [From Chain Of Custody, Pages 9-9]."
    c2 = grade_citations(bad, ["Cover Letter"], ["Cover Letter"])
    assert c2["citations_valid"] is False and c2["citations_correct"] is False

    c3 = grade_citations("No citation here.", ["Cover Letter"], ["Cover Letter"])
    assert c3["has_citation"] is False

    # Multi-citation answer.
    multi = "A [From Material Description, Pages 7-7] and B [From Bse/Tse Declaration, Pages 6-6]."
    c4 = grade_citations(multi, ["Material Description", "Bse/Tse Declaration"],
                         ["Material Description", "Bse/Tse Declaration"])
    assert c4["n_citations"] == 2 and c4["citations_valid"] and c4["citations_correct"]

    print("Metric self-test passed: retrieval, abstention, keyword grading and citation parsing all behave as specified.")


_selftest()

## Step 4 — Run the evaluation

Three passes: document processing (timed), retrieval alone, then the full pipeline end to
end. Every question is asked of both the digital and the scanned copy of its document.

In [ ]:
# ============================================
# EVAL STEP 4a: Run the corpus through the pipeline (timed)
# ============================================
#
# WHAT WE'RE DOING:
# Processing every file in the test corpus through the full pipeline and
# capturing a per-stage timing breakdown for each. Each file gets its own
# EnhancedDocumentStore so the four runs stay independent, and the
# classification output is checked against the document types the answer
# key says should be present.
#
# WHY THIS MATTERS:
# This produces two of the three required metric families at once. The
# System Performance numbers (OCR time, embedding time, indexing time)
# come from the timing breakdown. The classification check is the first
# accuracy signal: if the pipeline cannot find the Chain Of Custody
# document at all, no amount of retrieval tuning will answer questions
# about it -- an error that would otherwise be invisible behind a
# plausible-sounding answer drawn from the wrong document.
#
# WHAT YOU'LL SEE:
# Progress output per file, then a per-file summary table and a stage
# timing table. Expect the scanned variants to take substantially longer,
# because every page goes through Tesseract.
# ============================================

stores: Dict[str, EnhancedDocumentStore] = {}
processing_rows = []
stage_timings: Dict[str, pd.DataFrame] = {}
raw_stage_timings: Dict[str, Dict] = {}

for _, row in corpus_df.iterrows():
    fname, path = row["File"], row["path"]
    print(f"\n{'=' * 70}\n{fname}  ({row['Variant']})\n{'=' * 70}")

    store = EnhancedDocumentStore()
    reset_timing()
    t0 = time.time()
    success, stats = store.process_pdf(path, filename=fname, verbose=True)
    wall = time.time() - t0

    stage_timing, stage_counts = dict(TIMING), dict(TIMING_COUNTS)
    raw_stage_timings[fname] = {"timing": stage_timing, "counts": stage_counts, "wall": wall}
    stage_timings[fname] = timing_table(stage_timing, stage_counts, wall)

    if not success:
        print(f"FAILED: {stats.get('error')}")
        processing_rows.append({"File": fname, "Variant": row["Variant"], "Status": "FAILED"})
        continue

    stores[fname] = store

    gt_doc = next(d for d in GROUND_TRUTH["documents"] if d["file"] == fname)
    expected = set(gt_doc["expected_doc_types"])
    found = set(stats["document_types"])
    missing, spurious = sorted(expected - found), sorted(found - expected)

    processing_rows.append({
        "File": fname,
        "Variant": row["Variant"],
        "Status": "OK",
        "Pages": stats["total_pages"],
        "OCR pages": stats["pages_from_ocr"],
        "Docs found": stats["documents_found"],
        "Chunks": stats["total_chunks"],
        "Types found": len(found),
        "Types expected": len(expected),
        "Type recall": round(len(expected & found) / len(expected), 3) if expected else None,
        "Missed types": ", ".join(missing) or "-",
        "Spurious types": ", ".join(spurious) or "-",
        "Wall time (s)": round(wall, 1),
        "OCR time (s)": round(stage_timing.get("ocr", 0.0), 1),
    })

    if missing:
        print(f"\n  MISSED document types (expected but never classified): {missing}")
    if spurious:
        print(f"  SPURIOUS document types (classified but not expected): {spurious}")

processing_df = pd.DataFrame(processing_rows)
print("\n\nDocument processing summary")
display(processing_df)

print("\nStage timing breakdown per file")
for fname, tdf in stage_timings.items():
    print(f"\n{fname}")
    display(tdf)

In [ ]:
# ============================================
# CHECKLIST STEP 4a: Retrieval performance
# ============================================
# Runs every graded question through retrieval only -- no answer writing --
# against both the digital and the scanned copy of its document, and scores
# the ranked results against the answer key.
#
# Retrieval is measured separately from answering on purpose. An answer can
# never be right if the passage containing it was never found, so this tells
# us whether a wrong answer is a search problem or a writing problem.
#
# Metrics are computed at k = 1, 3 and 5 to show the trade-off: searching
# wider finds the answer more often but returns more noise with it.
#
# You'll see: a summary table per cutoff, a per-question table, and a list of
# any question where the right document was not found at all.
# ============================================

K_VALUES = [1, 3, 5]
K_PRIMARY = 3

# Map each digital source file to itself plus its scanned counterpart, so
# every question is asked of both modalities.
VARIANTS_OF = defaultdict(list)
for d in GROUND_TRUTH["documents"]:
    if d["variant"] == "digital":
        VARIANTS_OF[d["file"]].append(("Digital", d["file"]))
    else:
        VARIANTS_OF[d["derived_from"]].append(("Scanned", d["file"]))

retrieval_rows = []

for q in QUESTIONS:
    if not q["retrieval_graded"]:
        continue

    for variant, fname in VARIANTS_OF[q["source_file"]]:
        store = stores.get(fname)
        if store is None:
            continue

        # Recall@K denominator: how many chunks in this file belong to a
        # document type the answer key marks as holding the answer.
        type_counts = store.chunks_by_doc_type()
        total_relevant = sum(type_counts.get(dt, 0) for dt in q["relevant_doc_types"])

        t0 = time.time()
        retrieved = store.retriever.retrieve(q["question"], k=max(K_VALUES),
                                             auto_route=True, verbose=False)
        latency_ms = (time.time() - t0) * 1000

        retrieved_types = [c.doc_type for c, _ in retrieved]
        routing = store.retriever.last_routing or {}

        for k in K_VALUES:
            m = retrieval_metrics(retrieved_types, q["relevant_doc_types"], total_relevant, k)
            retrieval_rows.append({
                "ID": q["id"],
                "Variant": variant,
                "Category": q.get("category", "-"),
                "File": fname,
                "k": k,
                "Hit": m["hit"],
                "Precision@K": round(m["precision_at_k"], 3),
                "Recall@K": (round(m["recall_at_k"], 3)
                             if m["recall_at_k"] == m["recall_at_k"] else None),
                "RR": round(m["reciprocal_rank"], 3),
                "Relevant found": m["n_relevant_retrieved"],
                "Relevant in file": total_relevant,
                "Expected type(s)": ", ".join(q["relevant_doc_types"]),
                "Retrieved type(s)": ", ".join(retrieved_types[:k]) or "-",
                "Routed to": routing.get("predicted_type"),
                "Routing conf.": routing.get("confidence"),
                "Scope": store.retriever.last_search_scope,
                "Latency (ms)": round(latency_ms, 1),
            })

retrieval_df = pd.DataFrame(retrieval_rows)

if retrieval_df.empty:
    raise RuntimeError("No retrieval results. Check that the corpus processed successfully.")

summary = (retrieval_df.groupby(["k", "Variant"])
           .agg(**{"Queries": ("ID", "count"),
                   "Hit Rate": ("Hit", "mean"),
                   "Precision@K": ("Precision@K", "mean"),
                   "Recall@K": ("Recall@K", "mean"),
                   "MRR": ("RR", "mean")})
           .round(3).reset_index())

print(f"Retrieval metrics -- {retrieval_df['ID'].nunique()} graded questions "
      f"x {retrieval_df['Variant'].nunique()} input variants")
print("Auto-routing on, i.e. exactly how the UI behaves by default.\n")
display(summary)

print(f"\nPer-question detail at k={K_PRIMARY}")
display(retrieval_df[retrieval_df["k"] == K_PRIMARY][[
    "ID", "Variant", "Category", "Hit", "Precision@K", "Recall@K", "RR",
    "Expected type(s)", "Retrieved type(s)", "Routed to", "Routing conf.", "Scope"
]].reset_index(drop=True))

misses = retrieval_df[(retrieval_df["k"] == K_PRIMARY) & (~retrieval_df["Hit"])]
if misses.empty:
    print(f"\nNo retrieval misses at k={K_PRIMARY}.")
else:
    print(f"\nRetrieval misses at k={K_PRIMARY} -- the expected document type was not in "
          f"the top {K_PRIMARY} results:")
    for _, r in misses.iterrows():
        print(f"  {r['ID']} ({r['Variant']}): wanted [{r['Expected type(s)']}], "
              f"got [{r['Retrieved type(s)']}]; routed to {r['Routed to']} "
              f"(confidence {r['Routing conf.']}) via {r['Scope']}")

In [ ]:
# ============================================
# CHECKLIST STEP 4b: End-to-end accuracy
# ============================================
# Runs every question through the complete pipeline -- search, then write the
# answer -- on both the digital and the scanned copy, and grades four things:
#
#   Answer accuracy   Does the answer contain the facts the answer key says
#                     it must? (Skipped, not failed, where no key is set.)
#   Abstention        On the four trick questions whose answers are NOT in
#                     the documents, did the system say so instead of
#                     inventing something?
#   Citations         Two separate checks. Valid = the cited document was
#                     actually retrieved. Correct = it was the right one.
#                     An answer can cite a real but wrong source, which looks
#                     completely trustworthy to a reader.
#   Faithfulness      An LLM judge checks whether every claim in the answer
#                     appears in the retrieved text. This is the
#                     "factual consistency" figure on the metrics slide.
#
# Caveat worth stating out loud: the judge is the same model that wrote the
# answer, so it grades itself and will be generous. Treat it as a smoke
# detector, not an audit. Set RUN_LLM_JUDGE = False to skip it.
#
# You'll see: one line per question as it runs (expect 10-20s each on a T4),
# then the full results table.
# ============================================

RUN_LLM_JUDGE = True
K_EVAL = 4                 # matches the UI slider default
KEYWORD_THRESHOLD = 0.6    # share of answer-key keywords needed to count as correct


def judge_faithfulness(question, answer, retrieved):
    """Ask the LLM whether the answer is fully supported by its context.
    Returns SUPPORTED / PARTIAL / UNSUPPORTED, or None if unparseable."""
    if not retrieved:
        return None

    context = "\n\n".join(f"{format_citation_label(c, include_file=False)}\n{c.text}"
                          for c, _ in retrieved)

    prompt = f"""You are grading whether an answer is supported by its source context.

Context:
{context}

Question: {question}

Answer to grade:
{answer}

Does every factual claim in the answer appear in the context above?
Reply with ONE word:
SUPPORTED - every claim appears in the context
PARTIAL - some claims appear, others do not
UNSUPPORTED - the answer states facts that are not in the context

One word only."""

    try:
        verdict = generate_text(prompt, max_new_tokens=8, stage="llm_judge").strip().upper()
        for label in ("UNSUPPORTED", "SUPPORTED", "PARTIAL"):
            if label in verdict:
                return label
        return None
    except Exception as e:
        print(f"  judge error: {e}")
        return None


answer_rows = []
answer_records = []          # full answer text, kept for the results export

reset_timing()
qa_t0 = time.time()

for q in QUESTIONS:
    for variant, fname in VARIANTS_OF[q["source_file"]]:
        store = stores.get(fname)
        if store is None:
            continue

        result = store.query(q["question"], auto_route=True, k=K_EVAL, verbose=False)
        answer = result["answer"]
        retrieved_types = [s["doc_type"] for s in result["sources"]]

        grade = grade_answer(answer, q.get("expected_answer_keywords", []),
                             answerable=q["answerable"], threshold=KEYWORD_THRESHOLD)
        cites = grade_citations(answer, retrieved_types, q["relevant_doc_types"])

        faith = None
        if RUN_LLM_JUDGE:
            pairs = store.retriever.retrieve(q["question"], k=K_EVAL,
                                             auto_route=True, verbose=False)
            faith = judge_faithfulness(q["question"], answer, pairs)

        answer_rows.append({
            "ID": q["id"],
            "Variant": variant,
            "Category": q.get("category", "-"),
            "Answerable": q["answerable"],
            "Answer correct": grade["correct"],
            "Grading mode": grade["grading_mode"],
            "Keyword coverage": (round(grade["coverage"], 2)
                                 if grade["coverage"] == grade["coverage"] else None),
            "Has citation": cites["has_citation"],
            "Citations valid": cites["citations_valid"],
            "Citations correct": cites["citations_correct"],
            "Cited": ", ".join(cites["cited_types"]) or "-",
            "Retrieved": ", ".join(sorted(set(retrieved_types))) or "-",
            "Faithfulness": faith,
            "Mean similarity": round(result["confidence"], 3),
            "Chunks used": result["chunks_used"],
            "Retrieval (s)": result["retrieval_sec"],
            "Generation (s)": result["generation_sec"],
            "Response time (s)": round(result["retrieval_sec"] + result["generation_sec"], 2),
        })

        answer_records.append({
            "id": q["id"], "variant": variant, "question": q["question"],
            "source_file": fname, "answer": answer, "sources": result["sources"],
            "routing_predicted_type": result["routing_predicted_type"],
            "routing_confidence": result["routing_confidence"],
            "search_scope": result["search_scope"],
        })

        flag = {True: "correct", False: "WRONG", None: "ungraded"}[grade["correct"]]
        print(f"{q['id']} [{variant:8s}] {flag:8s} "
              f"cite_valid={str(cites['citations_valid']):5s} "
              f"cite_correct={str(cites['citations_correct']):5s} "
              f"faith={str(faith):11s} {result['retrieval_sec']:.2f}s + "
              f"{result['generation_sec']:.1f}s")

qa_wall = time.time() - qa_t0
qa_stage_timing, qa_stage_counts = dict(TIMING), dict(TIMING_COUNTS)

answers_df = pd.DataFrame(answer_rows)
print(f"\nCompleted {len(answers_df)} question runs in {qa_wall / 60:.1f} minutes")
display(answers_df)

### Routing ablation

Query routing is the main design decision here and it is not obviously a win: narrowing the
search improves precision when the guess is right and destroys recall when it is wrong.
This compares the shipped default against searching everything, and against a hand-picked
perfect filter as an upper bound.

In [ ]:
# ============================================
# EVAL STEP 4d: Routing ablation
# ============================================
#
# WHAT WE'RE DOING:
# Taking a sample of questions and answering each one three ways:
#   auto_route    -- the shipped default: the LLM predicts a document type
#                    and, if confident, searches only that type's index
#   global_search -- routing disabled, search all chunks
#   manual_filter -- the correct document type supplied by hand (an upper
#                    bound on what perfect routing could achieve)
#
# WHY THIS MATTERS:
# Routing is the main design decision in this pipeline, and it is not
# obviously a win: narrowing the search improves precision when the
# prediction is right and destroys recall when it is wrong. Comparing the
# three modes on the same questions is what turns "we added routing" into
# a measured claim, and the manual_filter column shows how much of the
# remaining gap is routing error rather than retrieval error.
#
# WHAT YOU'LL SEE:
# One row per question per mode, then a summary comparing hit rate and
# latency across the three modes.
# ============================================

# One single-source question, one multi-source question, and the vague one.
ABLATION_IDS = ["Q01", "Q04", "Q08", "Q10"]

ablation_rows = []

for q in QUESTIONS:
    if q["id"] not in ABLATION_IDS:
        continue
    store = stores.get(q["source_file"])
    if store is None:
        continue

    type_counts = store.chunks_by_doc_type()
    total_relevant = sum(type_counts.get(dt, 0) for dt in q["relevant_doc_types"])

    modes = {
        "auto_route": dict(auto_route=True, filter_type=None),
        "global_search": dict(auto_route=False, filter_type=None),
    }
    if q["relevant_doc_types"]:
        modes["manual_filter"] = dict(auto_route=False,
                                      filter_type=q["relevant_doc_types"][0])

    for mode_name, kwargs in modes.items():
        result = store.query(q["question"], filter_type=kwargs["filter_type"],
                             auto_route=kwargs["auto_route"], k=K_EVAL, verbose=False)
        retrieved_types = [s["doc_type"] for s in result["sources"]]

        m = (retrieval_metrics(retrieved_types, q["relevant_doc_types"],
                               total_relevant, K_EVAL)
             if q["retrieval_graded"] else None)

        ablation_rows.append({
            "ID": q["id"],
            "Mode": mode_name,
            "Hit": m["hit"] if m else None,
            "Precision@K": round(m["precision_at_k"], 3) if m else None,
            "RR": round(m["reciprocal_rank"], 3) if m else None,
            "Sources used": ", ".join(sorted(set(retrieved_types))),
            "Chunks used": result["chunks_used"],
            "Mean similarity": round(result["confidence"], 3),
            "Retrieval (s)": result["retrieval_sec"],
            "Generation (s)": result["generation_sec"],
            "Total (s)": round(result["retrieval_sec"] + result["generation_sec"], 2),
            "answer": result["answer"],
        })

        print(f"{q['id']} [{mode_name}]: sources={sorted(set(retrieved_types))}, "
              f"chunks={result['chunks_used']}, "
              f"similarity={result['confidence']:.2f}, "
              f"{result['retrieval_sec'] + result['generation_sec']:.1f}s")

ablation_df = pd.DataFrame(ablation_rows)

if not ablation_df.empty:
    print("\nAblation detail")
    display(ablation_df.drop(columns=["answer"]))

    ablation_summary = (ablation_df.groupby("Mode")
                        .agg(**{
                            "Questions": ("ID", "count"),
                            "Hit Rate": ("Hit", "mean"),
                            "Precision@K": ("Precision@K", "mean"),
                            "MRR": ("RR", "mean"),
                            "Mean chunks": ("Chunks used", "mean"),
                            "Mean total (s)": ("Total (s)", "mean"),
                        }).round(3).reset_index())
    print("\nAblation summary")
    display(ablation_summary)
else:
    ablation_summary = pd.DataFrame()
    print("No ablation results produced.")

## Step 5 — Results

In [ ]:
# ============================================
# CHECKLIST STEP 5: Show the results clearly
# ============================================
# One headline table, broken down the way the checklist asks for it:
# Digital / Scanned / Overall, so it is obvious at a glance both how well the
# pipeline does and where it struggles.
#
# Every row reports how many questions it was computed over. A number based
# on four control questions should not be read as though it came from
# thirty-two.
#
# You'll see: the headline table, then breakdowns by question type, then the
# performance-metrics block laid out to match the slide, then an explicit
# list of what failed.
# ============================================

def pct(x):
    """Percentage, or '-' when there is nothing to average."""
    return "-" if x is None or x != x else f"{x * 100:.0f}%"


def secs(x):
    return "-" if x is None or x != x else f"{x:.1f} sec"


def _slice(df, variant):
    return df if variant == "Overall" else df[df["Variant"] == variant]


VARIANT_COLS = ["Digital", "Scanned", "Overall"]
primary = retrieval_df[retrieval_df["k"] == K_PRIMARY]

# ---------------------------------------------------------------
# THE HEADLINE TABLE
# ---------------------------------------------------------------
rows = []

def add_row(label, fn, note=""):
    entry = {"Metric": label}
    for v in VARIANT_COLS:
        entry[v] = fn(v)
    entry["Based on"] = note
    rows.append(entry)


add_row("Answer accuracy",
        lambda v: pct(_slice(answers_df[answers_df["Answerable"] &
                                        answers_df["Answer correct"].notna()], v)["Answer correct"].mean()),
        f"{len(answers_df[answers_df['Answerable'] & answers_df['Answer correct'].notna()])} graded runs")

add_row("Correct refusal on trick questions",
        lambda v: pct(_slice(answers_df[~answers_df["Answerable"]], v)["Answer correct"].mean()),
        f"{len(answers_df[~answers_df['Answerable']])} control runs")

add_row("Citation accuracy (right source cited)",
        lambda v: pct(_slice(answers_df[answers_df["Citations correct"].notna()], v)["Citations correct"].mean()),
        f"{int(answers_df['Citations correct'].notna().sum())} cited answers")

add_row("Citation validity (source was retrieved)",
        lambda v: pct(_slice(answers_df[answers_df["Citations valid"].notna()], v)["Citations valid"].mean()),
        f"{int(answers_df['Citations valid'].notna().sum())} cited answers")

if answers_df["Faithfulness"].notna().any():
    add_row("Factual consistency (self-judged)",
            lambda v: pct((_slice(answers_df[answers_df["Faithfulness"].notna()], v)["Faithfulness"]
                           == "SUPPORTED").mean()),
            f"{int(answers_df['Faithfulness'].notna().sum())} judged answers")

add_row(f"Retrieval Recall@{K_PRIMARY}",
        lambda v: pct(_slice(primary, v)["Recall@K"].mean()),
        f"{primary['ID'].nunique()} graded questions")

add_row(f"Retrieval Hit Rate@{K_PRIMARY}",
        lambda v: pct(_slice(primary, v)["Hit"].mean()),
        f"{primary['ID'].nunique()} graded questions")

add_row("Mean Reciprocal Rank (MRR)",
        lambda v: f"{_slice(primary, v)['RR'].mean():.2f}",
        f"{primary['ID'].nunique()} graded questions")

_ok = processing_df[processing_df["Status"] == "OK"]
add_row("Document-type classification recall",
        lambda v: pct(_slice(_ok, v)["Type recall"].mean()),
        f"{len(_ok)} files")

add_row("Avg response time",
        lambda v: secs(_slice(answers_df, v)["Response time (s)"].mean()),
        f"{len(answers_df)} question runs")

add_row("Avg retrieval latency",
        lambda v: f"{_slice(answers_df, v)['Retrieval (s)'].mean() * 1000:.0f} ms",
        f"{len(answers_df)} question runs")

add_row("Avg LLM generation time",
        lambda v: secs(_slice(answers_df, v)["Generation (s)"].mean()),
        f"{len(answers_df)} question runs")

headline_df = pd.DataFrame(rows)[["Metric"] + VARIANT_COLS + ["Based on"]]

print("=" * 88)
print("PIPELINE EVALUATION RESULTS".center(88))
print("=" * 88)
print(f"Test set: {len(corpus_df)} documents ({int((corpus_df['Variant'] == 'Digital').sum())} digital, "
      f"{int((corpus_df['Variant'] == 'Scanned').sum())} scanned), "
      f"{int(corpus_df['Pages'].sum())} pages, {len(QUESTIONS)} questions, "
      f"{len(answers_df)} total question runs\n")
display(headline_df)

# ---------------------------------------------------------------
# WHERE IT STRUGGLES
# ---------------------------------------------------------------
by_category = (answers_df.groupby(["Category", "Variant"])
               .agg(**{"Runs": ("ID", "count"),
                       "Answer correct": ("Answer correct", "mean"),
                       "Citations correct": ("Citations correct", "mean"),
                       "Avg response (s)": ("Response time (s)", "mean")})
               .round(2).reset_index())
print("\nBy question type -- this is where the weaknesses show up")
display(by_category)

retr_by_cat = (primary.groupby(["Category", "Variant"])
               .agg(**{"Queries": ("ID", "count"),
                       "Hit Rate": ("Hit", "mean"),
                       f"Recall@{K_PRIMARY}": ("Recall@K", "mean"),
                       "MRR": ("RR", "mean")})
               .round(2).reset_index())
print(f"\nRetrieval by question type (k={K_PRIMARY})")
display(retr_by_cat)

# ---------------------------------------------------------------
# SYSTEM PERFORMANCE (metrics slide, bottom block)
# ---------------------------------------------------------------
combined_timing, combined_counts = defaultdict(float), defaultdict(int)
for entry in raw_stage_timings.values():
    for stage, s in entry["timing"].items():
        combined_timing[stage] += s
        combined_counts[stage] += entry["counts"].get(stage, 0)
for stage, s in qa_stage_timing.items():
    combined_timing[stage] += s
    combined_counts[stage] += qa_stage_counts.get(stage, 0)

ok_files = _ok
ocr_pages = int(ok_files["OCR pages"].sum()) if not ok_files.empty else 0
total_pages = int(ok_files["Pages"].sum()) if not ok_files.empty else 0

print("\nSystem performance")
perf_rows = [
    {"Measure": "Avg response time (retrieval + generation)",
     "Value": secs(answers_df["Response time (s)"].mean())},
    {"Measure": "Retrieval latency",
     "Value": f"{answers_df['Retrieval (s)'].mean() * 1000:.0f} ms"},
    {"Measure": "LLM generation",
     "Value": secs(answers_df["Generation (s)"].mean())},
    {"Measure": "OCR processing",
     "Value": (f"{combined_timing['ocr'] / ocr_pages:.1f} sec per scanned page"
               if ocr_pages else "not exercised")},
    {"Measure": "Document processing (end to end)",
     "Value": (f"{ok_files['Wall time (s)'].mean() / max(total_pages / len(ok_files), 1):.1f} sec per page"
               if not ok_files.empty else "-")},
    {"Measure": "Embedding",
     "Value": f"{combined_timing['embedding']:.1f} sec total for "
              f"{sum(len(s.chunks_metadata) for s in stores.values())} chunks"},
]
display(pd.DataFrame(perf_rows))

print("\nTotal time by pipeline stage (whole evaluation)")
display(timing_table(dict(combined_timing), dict(combined_counts)))

# ---------------------------------------------------------------
# FAILURES, NAMED
# ---------------------------------------------------------------
print("\n" + "=" * 88)
print("WHAT FAILED".center(88))
print("=" * 88)

wrong = answers_df[answers_df["Answer correct"] == False]
if wrong.empty:
    print("\nNo incorrect answers among graded questions.")
else:
    print(f"\nIncorrect answers ({len(wrong)}):")
    for _, r in wrong.iterrows():
        print(f"  {r['ID']} [{r['Variant']}] {r['Category']} -- graded by {r['Grading mode']}")

bad = answers_df[(answers_df["Citations valid"] == False) |
                 (answers_df["Citations correct"] == False)]
if bad.empty:
    print("\nNo citation problems detected.")
else:
    print(f"\nCitation problems ({len(bad)}):")
    for _, r in bad.iterrows():
        issue = ("cited a document that was never retrieved" if r["Citations valid"] == False
                 else "cited the wrong document")
        print(f"  {r['ID']} [{r['Variant']}]: {issue}; cited [{r['Cited']}], "
              f"retrieved [{r['Retrieved']}]")

no_cite = answers_df[answers_df["Has citation"] == False]
if not no_cite.empty:
    print(f"\nAnswers with no citation at all ({len(no_cite)}): "
          f"{', '.join(sorted(set(no_cite['ID'])))}")

missed = ok_files[ok_files["Missed types"] != "-"]
if not missed.empty:
    print("\nDocument types the classifier never found:")
    for _, r in missed.iterrows():
        print(f"  {r['File']}: {r['Missed types']}")

ungraded = answers_df[answers_df["Answer correct"].isna()]
if not ungraded.empty:
    print(f"\nNot graded for answer accuracy -- no answer key filled in yet "
          f"({ungraded['ID'].nunique()} questions): {', '.join(sorted(set(ungraded['ID'])))}")
    print("  These still count toward citation, faithfulness and retrieval metrics.")

In [ ]:
# ============================================
# Export the results
# ============================================
# Writes everything above to disk in three forms: results.md to read, CSVs
# to chart or paste into slides, and metrics.json holding the headline
# numbers in one place.
#
# The point of metrics.json is that the figures on the Pipeline Performance
# Metrics slide can be copied from a file rather than transcribed off the
# screen, where digits get dropped.
#
# You'll see: the list of files written, then the headline numbers printed
# in slide order.
# ============================================

OUT_DIR = "/content/eval_output"
os.makedirs(OUT_DIR, exist_ok=True)


def variant_split(df, col, agg="mean"):
    """Return {Digital, Scanned, Overall} for one column."""
    out = {}
    for v in ["Digital", "Scanned"]:
        sub = df[df["Variant"] == v][col].dropna()
        out[v] = round(float(getattr(sub, agg)()), 4) if len(sub) else None
    allv = df[col].dropna()
    out["Overall"] = round(float(getattr(allv, agg)()), 4) if len(allv) else None
    return out


# ---------------- CSVs ----------------
tables = {
    "01_test_set": corpus_df.drop(columns=["path"], errors="ignore"),
    "02_document_processing": processing_df,
    "03_retrieval_per_question": retrieval_df,
    "04_answers_per_question": answers_df,
    "05_routing_ablation": ablation_df.drop(columns=["answer"], errors="ignore"),
    "06_headline_results": headline_df,
    "07_by_question_type": by_category,
    "08_architecture_spec": architecture_df,
}
for name, df in tables.items():
    if isinstance(df, pd.DataFrame) and not df.empty:
        df.to_csv(f"{OUT_DIR}/{name}.csv", index=False)

# ---------------- metrics.json ----------------
graded_ans = answers_df[answers_df["Answerable"] & answers_df["Answer correct"].notna()]
controls = answers_df[~answers_df["Answerable"]]
cited = answers_df[answers_df["Citations valid"].notna()]

metrics = {
    "generated": datetime.now().isoformat(timespec="seconds"),
    "architecture": {
        "flow": PIPELINE_FLOW,
        "llm": LLM_MODEL_NAME,
        "embedding_model": EMBED_MODEL_NAME,
        "embedding_dim": int(embed_model.get_sentence_embedding_dimension()),
        "vector_store": "FAISS IndexFlatIP (cosine)",
        "ocr": f"Tesseract via pytesseract at {OCR_DPI} DPI",
        "chunk_size_words": CHUNK_SIZE_WORDS,
        "chunk_overlap_words": CHUNK_OVERLAP_WORDS,
        "top_k": K_EVAL,
        "routing_confidence_threshold": ROUTING_CONFIDENCE_THRESHOLD,
        "device": device,
    },
    "test_set": {
        "documents": int(len(corpus_df)),
        "digital_files": int((corpus_df["Variant"] == "Digital").sum()),
        "scanned_files": int((corpus_df["Variant"] == "Scanned").sum()),
        "total_pages": int(corpus_df["Pages"].sum()),
        "questions": len(QUESTIONS),
        "question_runs": int(len(answers_df)),
        "answerable": int(sum(q["answerable"] for q in QUESTIONS)),
        "unanswerable_controls": int(sum(not q["answerable"] for q in QUESTIONS)),
    },
    "retrieval": {},
    "end_to_end": {
        "answer_accuracy": variant_split(graded_ans, "Answer correct") if len(graded_ans) else None,
        "answer_accuracy_n": int(len(graded_ans)),
        "correct_abstention": variant_split(controls, "Answer correct") if len(controls) else None,
        "correct_abstention_n": int(len(controls)),
        "citation_present_rate": round(float((answers_df["Has citation"] == True).mean()), 4),
        "citation_validity": variant_split(cited, "Citations valid") if len(cited) else None,
        "citation_correctness": (variant_split(answers_df[answers_df["Citations correct"].notna()],
                                               "Citations correct")
                                 if answers_df["Citations correct"].notna().any() else None),
    },
    "system": {
        "avg_response_time_sec": variant_split(answers_df, "Response time (s)"),
        "avg_retrieval_latency_ms": {k: (round(v * 1000, 1) if v is not None else None)
                                     for k, v in variant_split(answers_df, "Retrieval (s)").items()},
        "avg_generation_sec": variant_split(answers_df, "Generation (s)"),
        "stage_totals_sec": {k: round(v, 2)
                             for k, v in sorted(combined_timing.items(), key=lambda x: -x[1])},
    },
}

if answers_df["Faithfulness"].notna().any():
    faith = answers_df[answers_df["Faithfulness"].notna()].copy()
    faith["_supported"] = (faith["Faithfulness"] == "SUPPORTED").astype(float)
    metrics["end_to_end"]["factual_consistency"] = variant_split(faith, "_supported")

for k in K_VALUES:
    sub = retrieval_df[retrieval_df["k"] == k]
    metrics["retrieval"][f"k={k}"] = {
        "hit_rate": variant_split(sub.assign(_h=sub["Hit"].astype(float)), "_h"),
        "precision_at_k": variant_split(sub, "Precision@K"),
        "recall_at_k": variant_split(sub, "Recall@K"),
        "mrr": variant_split(sub, "RR"),
        "queries": int(sub["ID"].nunique()),
    }

with open(f"{OUT_DIR}/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

# ---------------- results.md ----------------
def md_table(df):
    return df.to_markdown(index=False) + "\n\n"

md = [
    "# RAG Pipeline Evaluation Results\n\n",
    f"Generated: {metrics['generated']}  \n",
    f"Model: `{LLM_MODEL_NAME}` | Embeddings: `{EMBED_MODEL_NAME}` | Device: `{device}`\n\n",
    "## Headline results\n\n", md_table(headline_df),
    "## System architecture\n\n", f"`{PIPELINE_FLOW}`\n\n", md_table(architecture_df),
    "## Test set\n\n", md_table(tables["01_test_set"]),
    "## Document processing\n\n", md_table(processing_df),
    "## Retrieval performance\n\n", md_table(summary),
    f"### Per-question detail (k={K_PRIMARY})\n\n",
    md_table(retrieval_df[retrieval_df["k"] == K_PRIMARY].drop(columns=["Latency (ms)"],
                                                               errors="ignore")),
    "## End-to-end results\n\n", md_table(answers_df),
    "### By question type\n\n", md_table(by_category),
]

if not ablation_df.empty:
    md += ["## Routing ablation\n\n", md_table(ablation_summary),
           "### Detail\n\n", md_table(ablation_df.drop(columns=["answer"]))]

md += ["## Time by pipeline stage\n\n",
       md_table(timing_table(dict(combined_timing), dict(combined_counts))),
       "## Full answers\n\n"]

for rec in answer_records:
    md.append(f"**{rec['id']} [{rec['variant']}]: {rec['question']}**  \n")
    md.append(f"*File: {rec['source_file']} | scope: {rec['search_scope']}*\n\n")
    md.append(f"{rec['answer']}\n\n")

with open(f"{OUT_DIR}/results.md", "w") as f:
    f.write("".join(md))

print(f"Wrote to {OUT_DIR}:")
for fname in sorted(os.listdir(OUT_DIR)):
    print(f"  {fname}  ({os.path.getsize(os.path.join(OUT_DIR, fname)) / 1024:.1f} KB)")

print("\n" + "=" * 70)
print("NUMBERS FOR THE PIPELINE PERFORMANCE METRICS SLIDE")
print("=" * 70)
r = metrics["retrieval"].get(f"k={K_PRIMARY}", {})
e, sy = metrics["end_to_end"], metrics["system"]


def show(label, val, suffix="%"):
    if val is None:
        print(f"  {label}: not measured")
    elif isinstance(val, dict):
        print(f"  {label}: {val['Overall'] * 100:.1f}{suffix}"
              if suffix == "%" and val["Overall"] is not None else f"  {label}: {val['Overall']}")


print(f"\nRetrieval Performance (k={K_PRIMARY})")
for lbl, key in [("Recall@K", "recall_at_k"), ("Precision@K", "precision_at_k"),
                 ("Hit Rate", "hit_rate")]:
    show(lbl, r.get(key))
print(f"  MRR: {r.get('mrr', {}).get('Overall')}")

print("\nEnd-to-End Accuracy")
show(f"Answer Accuracy (n={e['answer_accuracy_n']})", e["answer_accuracy"])
show("Citation Accuracy", e["citation_correctness"])
show("Factual Consistency", e.get("factual_consistency"))

print("\nSystem Performance")
print(f"  Average Response Time: {sy['avg_response_time_sec']['Overall']} sec")
print(f"  Retrieval Latency: {sy['avg_retrieval_latency_ms']['Overall']} ms")
print(f"  LLM Generation: {sy['avg_generation_sec']['Overall']} sec")
print(f"  OCR Processing: {sy['stage_totals_sec'].get('ocr', 0)} sec total")

# Uncomment to download from Colab:
# from google.colab import files
# import shutil
# shutil.make_archive("/content/eval_output", "zip", OUT_DIR)
# files.download("/content/eval_output.zip")

### Optional: drafting the answer key

Answer accuracy needs a human to decide what facts a correct answer must contain. What can
be automated is finding the passage, so you only have to read it and type the facts that
matter.

In [ ]:
# ============================================
# OPTIONAL: Draft the answer key from the real documents
# ============================================
#
# WHAT WE'RE DOING:
# For every answerable question that has no expected_answer_keywords yet,
# printing the text the retriever surfaces from the expected document
# type, and writing a draft ground_truth_draft.json with an empty slot
# ready to fill.
#
# WHY THIS MATTERS:
# Keyword-based answer accuracy needs a human to decide what a correct
# answer must contain -- that judgement is the whole point of a ground
# truth and cannot be automated without the answer key grading itself.
# What can be automated is the tedious part: finding the passage, so you
# only have to read it and type the two or three facts that matter.
#
# HOW TO USE IT:
# Run this cell, read each printed passage, then edit ground_truth.json
# (or the generated draft) to add keywords like ["2-8", "celsius"].
# Re-run the ground-truth loading cell and then the evaluation cells to
# get keyword-graded answer accuracy. Everything else in the evaluation
# runs without this step.
#
# WHAT YOU'LL SEE:
# The retrieved passage for each unfilled question, and a written draft
# file path.
# ============================================

def bootstrap_answer_key(top_k: int = 2, chars: int = 700,
                         out_path: str = "/content/ground_truth_draft.json"):
    """Print candidate source text for every question lacking an answer key."""
    draft = json.loads(json.dumps(GROUND_TRUTH))  # deep copy
    pending = 0

    for q in draft["questions"]:
        if not q["answerable"] or q.get("expected_answer_keywords"):
            continue
        store = stores.get(q["source_file"])
        if store is None:
            continue

        pending += 1
        print(f"\n{'=' * 70}\n{q['id']}: {q['question']}")
        print(f"Expected source: {', '.join(q['relevant_doc_types']) or '(any)'}\n{'-' * 70}")

        # Search inside the expected document type where one is specified,
        # so the passage shown is the one the answer key should describe.
        filt = q["relevant_doc_types"][0] if q["relevant_doc_types"] else None
        hits = store.retriever.retrieve(q["question"], k=top_k,
                                        filter_doc_type=filt, auto_route=False, verbose=False)
        if not hits:
            print("  (nothing retrieved)")
            continue

        for chunk, score in hits:
            print(f"[{chunk.doc_type}, pages {chunk.pages_1indexed}, similarity {score:.2f}]")
            print(chunk.text[:chars].strip())
            print()

        q["expected_answer_keywords"] = []   # ready to fill

    with open(out_path, "w") as f:
        json.dump(draft, f, indent=2)

    print(f"\n{'=' * 70}")
    print(f"{pending} question(s) awaiting keywords. Draft written to {out_path}.")
    print("Fill in expected_answer_keywords there (or in ground_truth.json), then re-run")
    print("the ground-truth loading cell and the evaluation cells.")


# Uncomment to run:
# bootstrap_answer_key()
print("bootstrap_answer_key() is defined. Uncomment the call above to draft the answer key.")